大規模言語モデル入門Ⅱの10章を参考に評価用スクリプトの作成を行う。(とりあえず本の写経から)

### 10.2.3 llm-jp-evalで使用される評価指標
- 完全一致率(exact match ratio): 正解テキストと予測テキストが完全一致しているか否かを比較する指標。llm-jp-evalでは自然言語推論、多肢選択式質問応答、数学的推論で使用。

$\text{完全一致率}=\frac{\text{正解事例と予測事例の一致数}}{事例数}$

In [1]:
def calc_exact_match_ratio(trues: list[str], preds: list[str]) -> float:
    """
    完全一致率を計算する関数
    Args:
        trues (list[str]): 正解テキストのリスト
        preds (list[str]): 予測テキストのリスト
    """
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    
    # 正解テキストと予測テキストが一致していれば1、そうでなければ0
    num_exact_match = sum(
        1 if t == p else 0 for t, p in zip(trues, preds)
    )
    return num_exact_match / len(trues)

# 正解テキスト列
trues = ["entailment", "entailment", "contradiction"]
# 予測テキスト列
preds = ["entailment", "entailmen", "neutral"]
# 完全一致率を算出する
print("完全一致率:", calc_exact_match_ratio(trues, preds))

完全一致率: 0.3333333333333333


- 文字ベースF値: 正解テキストと予測テキストの文字の一致に基づいた適合率と再現率の調和平均。質問応答と機械読解で使用。

$\text{文字ベース適合率}=\frac{正解テキストと予測テキストで一致した文字数}{予測テキストの文字数}$

$\text{文字ベース再現率}=\frac{正解テキストと予測テキストで一致した文字数}{正解テキストの文字数}$

$\text{文字ベースF値}=\frac{2\cdot \text{文字ベース適合率}\cdot \text{文字ベース再現率}}{\text{文字ベース適合率}+\text{文字ベース再現率}}$

In [2]:
from collections import Counter

def calc_char_f1(trues: list[str], preds: list[str]) -> float:
    """
    文字ベースF値を計算する関数
    Args:
        trues (list[str]): 正解テキストのリスト
        preds (list[str]): 予測テキストのリスト
    """
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    
    char_f1_scores = []
    # 各事例ごとに文字ベースF値を算出する
    for t, p in zip(trues, preds):
        # 正解テキストと予測テキストのどちらかの文字がない場合
        if len(t) == 0 or len(p) == 0:
            # 別に0でよくない？
            char_f1_scores.append(float(t==p))
            break
        # 正解テキストと予測テキストの一致している文字を算出する
        common= Counter(list(t)) & Counter(list(p))
        num_same = sum(common.values())
        # 適合率の算出
        precision = num_same / len(p)
        # 再現率の算出
        recall = num_same / len(t)
        # F値の算出
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        char_f1_scores.append(f1_score)

    return sum(char_f1_scores) / len(char_f1_scores)

# 正解のテキスト列
trues = ["夏目漱石"]
# 予測のテキスト列
preds = ["夏目 漱石"]
# 文字ベースF値を算出する
print("文字ベースF値:", calc_char_f1(trues, preds))

文字ベースF値: 0.888888888888889


- 集合ベースF値: 正解が集合で与えられるタスクの性能評価に使う指標。正解集合と予測集合でF1スコアを算出して計算。主にエンティティ極性分析と基礎解析で使用。
(集合: 1つの設問に対して複数の回答テキストが存在する際の回答テキスト集合)

$\text{集合ベース適合率}=\frac{正解集合と予測集合で一致した要素数}{予測テキストの要素数}$

$\text{集合ベース再現率}=\frac{正解集合と予測集合で一致した要素数}{正解テキストの要素数}$

$\text{集合ベースF値}=\frac{2\cdot \text{集合ベース適合率}\cdot \text{集合ベース再現率}}{\text{集合ベース適合率}+\text{集合ベース再現率}}$

In [3]:
def calc_set_f1(trues: list[set[str]], preds: list[set[str]]) -> float:
    """
    集合ベースF値を計算する関数
    Args:
        trues (list[set[str]]): 正解テキスト集合のリスト
        preds (list[set[str]]): 予測テキスト集合のリスト
    """
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    
    set_f1_scores = []
    # 事例単位で処理
    for t, p in zip(trues, preds):
        # 正解データを行ごとに分割し、集合にする
        split_t = {x.strip() for x in t.split("\n")}
        # 予測データを行ごとに分割し、集合にする
        split_p = {x.strip() for x in p.split("\n")}
        # 適合率の算出
        precision = sum(1 if y in split_t else 0 for y in split_p) / len(split_p)
        # 再現率の算出
        recall = sum(1 if y in split_p else 0 for y in split_t) / len(split_t)
        # F値の算出
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        set_f1_scores.append(f1_score)
    
    return sum(set_f1_scores) / len(set_f1_scores)

# 正解集合
trues = [
    "営業収益 positive\n"
    "純営業収益 positive\n"
    "経常利益 positive\n"
    "当期純利益 positive"
]
# 予測集合
preds = ["営業収益 positive\n純営業収益 negative\n経常利益 positive"]
# 集合ベースF値を算出する
print("集合ベースF値:", calc_set_f1(trues, preds))

集合ベースF値: 0.5714285714285715


- 相関係数: ピアソンの積率相関係数とスピアマンの順位相関係数がある。実数値でないと算出できないので注意。

In [4]:
import math
from scipy.stats import pearsonr, spearmanr

def calc_pearsonr(trues: list[float], preds: list[float]) -> float:
    """
    ピアソンの積率相関係数を計算する関数
    Args:
        trues (list[float]): 正解値のリスト
        preds (list[float]): 予測値のリスト
    """
    scores = pearsonr(
        list(map(float, trues)), list(map(float, preds))
    ).statistic

    return 0.0 if math.isnan(scores) else scores

def calc_spearmanr(trues: list[float], preds: list[float]) -> float:
    """
    スピアマンの順位相関係数を計算する関数
    Args:
        trues (list[float]): 正解値のリスト
        preds (list[float]): 予測値のリスト
    """
    scores = spearmanr(
        list(map(float, trues)), list(map(float, preds))
    ).statistic

    return 0.0 if math.isnan(scores) else scores

trues = ["1.2", "2.2", "3.3", "4.9"]
preds = ["1.1", "4.1", "4.0", "5.0"]
print("ピアソンの積率相関係数:", calc_pearsonr(trues, preds))
print("スピアマンの順位相関係数:", calc_spearmanr(trues, preds))

ピアソンの積率相関係数: 0.8514063149390616
スピアマンの順位相関係数: 0.7999999999999999


LLM-jp-3.1-1.8Bと適当なデータセットで検証。

In [5]:
from datasets import load_dataset
dataset_name = "elyza/JaMARD"
# 成形前のデータを取得
# 成形後のデータを取得: load_dataset(dataset_name, trust_remote_code=True)
datasets = load_dataset(dataset_name, trust_remote_code=True, name="raw")
train_dataset = datasets["train"]
val_dataset = datasets["validation"]
test_dataset = load_dataset(dataset_name, trust_remote_code=True,  split="test[:100]", name="raw")
print(val_dataset)

Dataset({
    features: ['gold_answer', 'instruction', 'responses', 'accuracy>0', 'accuracy', 'true_responses', 'false_responses', 'answers', 'source'],
    num_rows: 1995
})


データセットは1つの問い(instruction)に対して、正解としている回答(true_response)と府世界とする回答(false_response)が複数存在している。したがって、最終的な回答もその数だけある。

In [6]:
test_dataset[0]["instruction"]

'ジェーンETのダックは毎日16個の卵を産みます。彼女は毎朝の朝食に3個を食べ、友人のために毎日マフィンを作るのに4個使います。彼女は農民市場で残りの卵を毎日2ドルの価格で販売します。彼女は農民市場で毎日何ドル稼いでいますか？'

In [7]:
from pprint import pprint

"""
def convert_data_format(data: dict[str, str]) -> dict[str, list]:
    データ形式を変換する関数
    Args:
        data (dict[str, str]): 元のデータ形式
    Returns:
        dict[str, list]: 変換後のデータ形式
    data["input"] = (
        f"質問: {data['question']}\n"
        f"選択肢: 1. {data['choice0']}, 2. {data['choice1']}, 3. {data['choice2']}, 4. {data['choice3']}\n"

    )
    data["output"] = data["label"]
    return data
"""
def convert_data_format(data: dict[str, str]) -> dict[str, list]:
    """
    データ形式を変換する関数
    Args:
        data (dict[str, str]): 元のデータ形式
    Returns:
        dict[str, list]: 変換後のデータ形式
    """
    # 問題文の抽出
    data["input"] = (
        f"問題:{data["instruction"]}"
    )
    # 正解の抽出
    # 複数個存在するので、最初のものを採用する
    data["output"] = data["answers"][0]
    return data

# 訓練セットをシャッフルする
train_dataset = train_dataset.shuffle()
# 訓練セットの前処理をする
train_dataset = train_dataset.map(convert_data_format)
# 四つのfew-shot事例を取得する
few_shots = list(train_dataset)[:4]
# 検証セットの前処理をする
val_dataset = val_dataset.map(convert_data_format)
# テストセットの前処理をする
test_dataset = test_dataset.map(convert_data_format)
pprint(list(test_dataset[0]))

Map:   0%|          | 0/17897 [00:00<?, ? examples/s]

['gold_answer',
 'instruction',
 'responses',
 'accuracy>0',
 'accuracy',
 'true_responses',
 'false_responses',
 'answers',
 'source',
 'input',
 'output']


In [8]:
# プロンプトテンプレートの作成
"""
def create_prompt_template(
        instruction: str, few_shots: list[dict[str, str]] | None = None
) -> str:
    プロンプトテンプレートを作成する関数
    Args:
        instruction (str): タスクの指示文
        few_shots (list[dict[str, str]] | None): few-shot事例のリスト
    Returns:
        str: プロンプトテンプレート
    prompt_template = (
        "以下は、タスクを説明する指示と、"
        "文脈のある入力の組み合わせです。"
        "要求を適切に満たす応答を書きなさい。\n\n"
    )
    prompt_template += f"### 指示: \n{instruction}\n\n"
    if few_shots is not None:
        for few_shot in few_shots:
            prompt_template += f"### 入力:\n{few_shot["input"]}\n\n"
            prompt_template += f"### 応答:\n{few_shot["output"]}\n\n"
    prompt_template += "### 入力:\n{input}\n\n"
    prompt_template += "### 応答:\n"
    return prompt_template
"""
def create_prompt_template(
        instruction: str, few_shots: list[dict[str, str]] | None = None
) -> str:
    """
    プロンプトテンプレートを作成する関数
    Args:
        instruction (str): タスクの指示文
        few_shots (list[dict[str, str]] | None): few-shot事例のリスト
    Returns:
        str: プロンプトテンプレート
    """
    prompt_template = (
        "以下は、タスクを説明する指示と、"
        "文脈のある入力の組み合わせです。"
        "要求を適切に満たす応答を書きなさい。\n\n"
    )
    prompt_template += f"### 指示: \n{instruction}\n\n"
    if few_shots is not None:
        for few_shot in few_shots:
            prompt_template += f"### 入力:\n{few_shot["input"]}\n\n"
            prompt_template += f"### 応答:\n{few_shot["output"]}\n\n"
    prompt_template += "### 入力:\n{input}\n\n"
    prompt_template += "### 応答:\n"
    return prompt_template
#instruction = "質問と回答の選択肢を入力として受け取り、選択肢から回答を選択してください。なお、回答は選択肢の番号（例：0）でするものとします。 回答となる数値をint型で返し、他には何も含めないことを厳守してください。"
instruction = "数学の問題を入力として受け取り、その問題の回答を出力してください。なお、回答には数値以外は含めないことを厳守してください。ただし、特殊な記号が必要な場合はその限りではありません。"
prompt_template = create_prompt_template(instruction, few_shots)
print(prompt_template)

以下は、タスクを説明する指示と、文脈のある入力の組み合わせです。要求を適切に満たす応答を書きなさい。

### 指示: 
数学の問題を入力として受け取り、その問題の回答を出力してください。なお、回答には数値以外は含めないことを厳守してください。ただし、特殊な記号が必要な場合はその限りではありません。

### 入力:
問題:アリスの白水流レジャーチームには40人の学生と10人の教師（彼女を含む）がいます。彼女には20つの救生衣が用意されています。20％の学生が自らの救生衣を持参しています。アリスはクラス全員が救生衣を持っているためには、さらにいくつの救生衣を用意する必要がありますか？

### 応答:
20

### 入力:
問題:$1-(1+(1-(1+(1-x))))$ をシンプルに変形せよ。

### 応答:
$2-x$

### 入力:
問題:「45,520」の桁を並べ替えて5桁の数を形成する方法は何通りですか？（注意: 数字は0から始まることはできません。）

### 応答:
3024

### 入力:
問題:トニーアさんはレモンジューススタンドをオープンし、小さなカップ、中サイズのカップ、そして大きなカップのレモンジュースを1ドル、2ドル、そして3ドルで販売しています。営業終了時に彼女は50ドルを稼いだとします。彼女が在庫を再確認したところ、小さなレモンジュースを11ドル、中サイズのレモンジュースを24ドル販売したことに気付きました。トニーアさんは大きなレモンジュースをいくつ販売したでしょうか？

### 応答:
5

### 入力:
{input}

### 応答:



パイプラインの作成

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_name = "llm-jp/llm-jp-3.1-13b-instruct4"
# Tokenizerの読み込み
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 量子化の設定
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_type=torch.bfloat16
)
# モデルの読み込み
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=True,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00006-of-00006.safetensors:   0%|          | 0.00/2.71G [00:00<?, ?B/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [10]:
# テキスト生成用のパラメータを指定
generation_config = {
    "max_new_tokens": 16, # 生成する最大トークン数
    "top_p": 1.0, # top-pサンプリング
    "repetition_penalty": 1.0 # 繰り返しペナルティ
}
# pipelineの作成
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config
)

Device set to use cuda:0


In [11]:
# 質問の回答を生成
from datasets import Dataset
from tqdm import tqdm
from transformers import TextGenerationPipeline

def generate_answers(
        text_generation_pipeline: TextGenerationPipeline,
        dataset: Dataset,
        prompt_template: str,
) -> list[dict[str, str]]:
    """
    質問の回答を生成する関数
    Args:
        text_generation_pipeline (TextGenerationPipeline): テキスト生成パイプライン
        dataset (Dataset): データセット
        prompt_template (str): プロンプトテンプレート
    Returns:
        list[dict[str, str]]: 生成された回答のリスト
    """
    results = []
    for data in tqdm(dataset):
        # プロンプトテンプレートの{input}部分を質問テキストに置換
        prompt = prompt_template.format(input=data["input"])
        # 質問の回答を生成
        output = text_generation_pipeline(prompt)
        # プロンプト部分を削除して予測部分のみを抽出
        generated_text = output[0]["generated_text"].replace(
            prompt, ""
        )
        # 複数行出力された場合、最初の行のみを抽出
        pred_label = generated_text.split("\n")[0].strip()
        results.append(
            {
                "input": data["input"],
                "true_label": str(data["output"]),
                "pred_label": str(pred_label),
            }
        )
    return results

# 検証セットに対して質問の回答を生成する
results1 = generate_answers(
    text_generation_pipeline, test_dataset, prompt_template
)
pprint(results1[:3])

100%|██████████| 100/100 [00:50<00:00,  1.97it/s]

[{'input': '問題:ジェーンETのダックは毎日16個の卵を産みます。彼女は毎朝の朝食に3個を食べ、友人のために毎日マフィンを作るのに4個使います。彼女は農民市場で残りの卵を毎日2ドルの価格で販売します。彼女は農民市場で毎日何ドル稼いでいますか？',
  'pred_label': '5',
  'true_label': '18'},
 {'input': '問題:ローブは2枚の青い繊維とそれと同じ半分の白い繊維が必要です。合計すると何枚の繊維が必要ですか？',
  'pred_label': '3',
  'true_label': '3'},
 {'input': '問題:ジョシュさんは「家を買う」を試みます。彼は80,000ドルの家を購入し、その後50,000ドルの修理を行います。これにより、家全体の価値が150％増加しました。彼が得た利益はどれくらいですか？',
  'pred_label': '10000',
  'true_label': '65'}]


In [12]:
for r in results1[:10]:
    print(f"入力文: {r['input']}")
    print(f"正解ラベル: {r['true_label']}")
    print(f"予測ラベル: {r['pred_label']}")
    print("-----")

入力文: 問題:ジェーンETのダックは毎日16個の卵を産みます。彼女は毎朝の朝食に3個を食べ、友人のために毎日マフィンを作るのに4個使います。彼女は農民市場で残りの卵を毎日2ドルの価格で販売します。彼女は農民市場で毎日何ドル稼いでいますか？
正解ラベル: 18
予測ラベル: 5
-----
入力文: 問題:ローブは2枚の青い繊維とそれと同じ半分の白い繊維が必要です。合計すると何枚の繊維が必要ですか？
正解ラベル: 3
予測ラベル: 3
-----
入力文: 問題:ジョシュさんは「家を買う」を試みます。彼は80,000ドルの家を購入し、その後50,000ドルの修理を行います。これにより、家全体の価値が150％増加しました。彼が得た利益はどれくらいですか？
正解ラベル: 65
予測ラベル: 10000
-----
入力文: 問題:ジェイムズさんは1週間に3回3つのスプリントを走ります。彼は1回のスプリントで60メートルを走ります。1週間に彼が走る合計メートル数はいくつですか？
正解ラベル: 540
予測ラベル: 432
-----
入力文: 問題:毎日、ウェンディは自分の鶏たちに種類、ミールワーム、野菜を含むミックスされた鶏用飼料を3カップずつ与え、それにより鶏たちを健康に保つために必要な栄養を提供します。鶏たちに与える飼料は3回に分けて与えられます。朝に、ウェンディは自分の鶏たちに15カップの飼料を与えます。午後に、ウェンディは鶏たちにさらに25カップの飼料を与えます。一日の最終的な飼料の与え方、すなわち、ウェンディの鶏の群れの大きさが20羽の場合、ウェンディは何カップの飼料を鶏たちに与える必要がありますか？
正解ラベル: 2
予測ラベル: 3
-----
入力文: 問題:クライアールさんは新しいアパートでガラスを購入するために店に行きました。一つのガラスは5ドルですが、二つ目のガラスは価格の60%になります。クライアールさんは16個のガラスを購入したいです。それらを購入するために必要な金額はいくつですか？
正解ラベル: 78
予測ラベル: 120
-----
入力文: 問題:トゥールーズには、チャールストンの羊の数が2倍です。チャールストンには、シカゴの羊の数が4倍です。シカゴが20頭の羊を持っている場合、トゥールーズ、チャールストン、そしてシカゴの羊の合計数はいく

In [13]:
# 完全一致率を算出する
true_labels1 = [r["true_label"] for r in results1]
pred_labels1 = [r["pred_label"] for r in results1]
score = calc_exact_match_ratio(true_labels1, pred_labels1)
print("完全一致率: ", score)

完全一致率:  0.06
